In [ ]:
import os, json, re
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/Users/nick/Desktop/intern/POLYMARKET')
os.chdir(BASE)

con = duckdb.connect()
con.execute("SET threads=4; SET memory_limit='3GB'; SET preserve_insertion_order=false;")

# ── Source ──────────────────────────────────────────────────────────────────
# '1s'      → elon_ticks_1s.parquet     ← recommended: 1-second bucketed,
#              price_change + last_trade_price only. Run create_1s_ticks.py first.
# 'master'  → elon_tweet_ticks.parquet  ← full 1.36 B row archive (slower)
# 'staging' → staging/src_*.parquet     ← use while scraper is mid-run
SOURCE = '1s'
# ──────────────────────────────────────────────────────────────────────────
print(f'Source: {SOURCE}')

In [ ]:
# ── Lookup tables ──────────────────────────────────────────────────────────

clob = json.loads((BASE / 'all_elon_cids_2026.json').read_text())
mids = json.loads((BASE / 'market_ids.json').read_text())

# Only keep Elon tweet COUNT markets (elon-musk-of-tweets-*)
cid_to_slug = {
    m['condition_id']: m['market_slug']
    for m in clob['markets']
    if m.get('condition_id') and m.get('market_slug')
    and re.match(r'elon-musk-of-tweets-', m['market_slug'])
}
print(f"Elon tweet count condition_ids: {len(cid_to_slug)}")

def slug_to_bucket(slug):
    m = re.search(r'(\d+-\d+)$', slug)
    if m: return m.group(1)
    m = re.search(r'(\d+plus)$', slug)
    if m: return m.group(1)
    return None

def slug_to_market(slug):
    m = re.search(
        r'of-tweets-(january|february|march|april|may|june|july|august|'
        r'september|october|november|december)-(\d{4})-', slug)
    if m: return f"{m.group(1).capitalize()} {m.group(2)} monthly"
    m = re.search(r'of-tweets-([a-z]+-\d+)-([a-z]+-\d+)', slug)
    if m: return f"{m.group(1)} – {m.group(2)} weekly"
    return slug

def bucket_bounds(b):
    """Parse 'lo-hi' or 'Xplus' bucket string → (lo, hi, mid)."""
    if b is None:
        return None, None, None
    m = re.match(r'^(\d+)-(\d+)$', str(b))
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        return lo, hi, (lo + hi) / 2
    m = re.match(r'^(\d+)plus$', str(b))
    if m:
        lo = float(m.group(1))
        return lo, None, None   # hi/mid filled in below
    return None, None, None

# Build bucket_labels: one row per condition_id
rows = []
for cid, slug in cid_to_slug.items():
    b  = slug_to_bucket(slug)
    mn = slug_to_market(slug)
    lo, hi, mid = bucket_bounds(b)
    if b and lo is not None:
        rows.append({'condition_id': cid, 'bucket': b, 'market_name': mn,
                     'bucket_lo': lo, 'bucket_hi': hi, 'bucket_mid': mid})

bucket_df = pd.DataFrame(rows)

# Fill open-ended "Xplus" hi/mid using the median bucket width per market
for mn, grp in bucket_df.groupby('market_name'):
    widths = (grp['bucket_hi'] - grp['bucket_lo']).dropna()
    bw = widths.median() if len(widths) > 0 else 20.0
    mask = bucket_df['market_name'].eq(mn) & bucket_df['bucket_hi'].isna()
    bucket_df.loc[mask, 'bucket_hi']  = bucket_df.loc[mask, 'bucket_lo'] + bw
    bucket_df.loc[mask, 'bucket_mid'] = bucket_df.loc[mask, 'bucket_lo'] + bw / 2

con.execute("CREATE OR REPLACE TEMP TABLE bucket_labels AS SELECT * FROM bucket_df")
print(f"bucket_labels: {len(bucket_df)} rows  ({bucket_df['market_name'].nunique()} markets)")

# Known YES/NO token sides from market_ids.json (includes condition_id)
known_rows = []
for m in mids:
    cid = m.get('contract_address', '')
    if m.get('yes_token'): known_rows.append({'asset_id': m['yes_token'], 'condition_id': cid, 'side': 'YES'})
    if m.get('no_token'):  known_rows.append({'asset_id': m['no_token'],  'condition_id': cid, 'side': 'NO'})
known_sides_df = pd.DataFrame(known_rows)
con.execute("CREATE OR REPLACE TEMP TABLE known_sides AS SELECT * FROM known_sides_df")
print(f"known_sides:   {len(known_sides_df)} tokens  ({known_sides_df['side'].value_counts().to_dict()})")

ELON_CID_SQL = '(' + ', '.join(f"'{c}'" for c in cid_to_slug) + ')'

In [ ]:
# ── Build OHLCV directly in DuckDB — no raw ticks in pandas ────────────────
#
# All aggregation runs inside DuckDB (bounded by memory_limit).
# Python only receives the small OHLCV result (~thousands of rows).
#
# Source schema differences:
#   master / staging  → raw tick columns: best_bid, best_ask, price, size, event_type, side
#   1s                → pre-bucketed:     best_bid, best_ask, trade_size, trade_price, n_quotes, n_trades

FREQ = '1 hour'   # output candle frequency: 'hour', 'day', '30 minutes', etc.

if SOURCE == '1s':
    src_path = str(BASE / 'elon_ticks_1s.parquet')
    if not Path(src_path).exists():
        raise FileNotFoundError(
            'elon_ticks_1s.parquet not found. '
            'Run: python create_1s_ticks.py'
        )
    # 1s file already has one row per (condition_id, asset_id, second).
    # best_bid/best_ask are the last quote in that second.
    # trade_size is summed trade volume in that second.
    src_query = f"""
        SELECT timestamp, condition_id, asset_id,
               best_bid, best_ask,
               trade_size                         AS size,
               CASE WHEN n_trades > 0 THEN 1
                    ELSE 0 END                    AS is_trade
        FROM   read_parquet('{src_path}')
        WHERE  condition_id IN {ELON_CID_SQL}
          AND  best_bid IS NOT NULL
    """
elif SOURCE == 'master':
    src_path = str(BASE / 'elon_tweet_ticks.parquet')
    if not Path(src_path).exists():
        raise FileNotFoundError('elon_tweet_ticks.parquet not found.')
    src_query = f"""
        SELECT timestamp, condition_id, asset_id,
               best_bid, best_ask,
               size,
               CASE WHEN event_type = 'last_trade_price' THEN 1
                    ELSE 0 END                    AS is_trade
        FROM   read_parquet('{src_path}')
        WHERE  condition_id IN {ELON_CID_SQL}
          AND  event_type   IN ('price_change', 'last_trade_price')
          AND  best_bid IS NOT NULL
    """
elif SOURCE == 'staging':
    all_staging = sorted(BASE.glob('staging/src_*.parquet'))
    if not all_staging:
        raise FileNotFoundError('No staging files. Switch SOURCE to "1s" or "master".')
    valid = [f for f in all_staging if f.stat().st_size > 8]
    file_list = '[' + ', '.join(f"'{f}'" for f in valid) + ']'
    src_query = f"""
        SELECT timestamp, condition_id, asset_id,
               best_bid, best_ask,
               size,
               CASE WHEN event_type = 'last_trade_price' THEN 1
                    ELSE 0 END                    AS is_trade
        FROM   read_parquet({file_list}, union_by_name=true)
        WHERE  condition_id IN {ELON_CID_SQL}
          AND  event_type   IN ('price_change', 'last_trade_price')
          AND  best_bid IS NOT NULL
    """
else:
    raise ValueError(f"Unknown SOURCE {SOURCE!r}. Use '1s', 'master', or 'staging'.")

ohlcv_df = con.execute(f"""
WITH
src AS ({src_query}),

-- ── Identify YES token per condition_id ─────────────────────────────────
avg_mid AS (
    SELECT condition_id, asset_id,
           AVG((best_bid + best_ask) / 2.0) AS avg_mid
    FROM   src
    GROUP  BY condition_id, asset_id
),
ranked AS (
    SELECT condition_id, asset_id,
           RANK() OVER (PARTITION BY condition_id ORDER BY avg_mid) AS rk
    FROM   avg_mid
),
known_yes_cids AS (
    SELECT DISTINCT condition_id FROM known_sides WHERE side = 'YES'
),
yes_tokens AS (
    SELECT asset_id, condition_id FROM known_sides WHERE side = 'YES'
    UNION ALL
    SELECT r.asset_id, r.condition_id FROM ranked r
    WHERE  r.rk = 1
      AND  r.condition_id NOT IN (SELECT condition_id FROM known_yes_cids)
),

-- ── Hourly OHLCV of YES mid-price ────────────────────────────────────────
hourly AS (
    SELECT
        s.condition_id,
        DATE_TRUNC('hour', s.timestamp)              AS ts,
        arg_min((s.best_bid + s.best_ask) / 2.0, s.timestamp) AS open,
        MAX((s.best_bid + s.best_ask) / 2.0)         AS high,
        MIN((s.best_bid + s.best_ask) / 2.0)         AS low,
        arg_max((s.best_bid + s.best_ask) / 2.0, s.timestamp) AS close,
        AVG(s.best_ask - s.best_bid)                 AS spread,
        SUM(s.size * s.is_trade)                     AS volume
    FROM   src s
    JOIN   yes_tokens yt ON s.condition_id = yt.condition_id
                        AND s.asset_id     = yt.asset_id
    GROUP  BY s.condition_id, DATE_TRUNC('hour', s.timestamp)
)

SELECT
    h.ts             AS timestamp,
    h.condition_id,
    bl.bucket,
    bl.market_name,
    bl.bucket_lo,
    bl.bucket_hi,
    bl.bucket_mid,
    h.open, h.high, h.low, h.close, h.spread,
    COALESCE(h.volume, 0) AS volume
FROM       hourly        h
JOIN       bucket_labels bl ON h.condition_id = bl.condition_id
ORDER BY   bl.market_name, bl.bucket_mid, h.ts
""").df()

ohlcv_df['timestamp'] = pd.to_datetime(ohlcv_df['timestamp'], utc=True)

print(f'Source:       {SOURCE}')
print(f'OHLCV ({FREQ}): {len(ohlcv_df):,} candles  |  {ohlcv_df["bucket"].nunique()} buckets  |  {ohlcv_df["market_name"].nunique()} markets')
print(f'RAM:          {ohlcv_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Date range:   {ohlcv_df["timestamp"].min()} → {ohlcv_df["timestamp"].max()}')
print(f'\nMarkets:')
print(ohlcv_df.groupby('market_name')['bucket'].nunique().rename('n_buckets').to_string())
ohlcv_df.head(5)

In [ ]:
# ohlcv_df already has all enriched columns from the DuckDB query above.
# (bucket, market_name, bucket_lo, bucket_hi, bucket_mid, open/high/low/close, spread, volume)
print(ohlcv_df.dtypes)
print(f'\n{len(ohlcv_df):,} candles ready.')

In [ ]:
# orderbook_df / trades_df are no longer needed — OHLCV and volume were
# computed in DuckDB. The downstream PCA cells use ohlcv_df directly.
print(f'ohlcv_df: {len(ohlcv_df):,} rows  |  {ohlcv_df["market_name"].nunique()} markets  |  {ohlcv_df["bucket"].nunique()} buckets')
print(f'Total USDC volume: ${ohlcv_df["volume"].sum():,.0f}')

In [ ]:
# OHLCV is already in ohlcv_df (built in cell-load). Quick sanity check:
print(f'OHLCV ({FREQ}): {len(ohlcv_df):,} candles  |  {ohlcv_df["bucket"].nunique()} buckets')
ohlcv_df.head(8)

In [ ]:
# ── Data quality checks ────────────────────────────────────────────────────
print('=== ohlcv_df quality ===')
print(f'  Negative spread:   {(ohlcv_df["spread"] < 0).sum()}')
print(f'  close out [0,1]:   {((ohlcv_df["close"] < 0) | (ohlcv_df["close"] > 1)).sum()}')
print(f'  NaN close:         {ohlcv_df["close"].isna().sum()}')

print('\n=== Probability sum per market (last candle close) ===')
print('  Should be ~1.0 across all buckets for the same market.')
print('  >1 = overpriced (sellers edge), <1 = underpriced (buyers edge)')
prob_sum = (
    ohlcv_df
    .sort_values('timestamp')
    .groupby(['market_name', 'bucket'])['close'].last()
    .reset_index()
    .groupby('market_name')['close'].sum()
)
print(prob_sum.to_string())

## PCA on Implied Tweet-Count Distribution

Each market (e.g. "May 2026 monthly") has N buckets covering non-overlapping tweet-count ranges.  
At each timestamp, the YES close prices ≈ P(tweets ∈ bucket_i) — a probability distribution over outcomes.

**Why not PCA on raw returns?**  
The N prices sum to 1 (simplex constraint), so one dimension is always redundant.  
Standard PCA on correlated prices mixes the signal with the constraint artefact.

**What we do instead:**  
Apply the **Centered Log-Ratio (CLR)** transform, which maps the simplex to ℝᴺ (rank N−1).  
Then PCA on the CLR-transformed matrix gives genuinely orthogonal components:
- **PC1** → location shift (market's expected tweet count moving up/down)
- **PC2** → spread / uncertainty (distribution widening or narrowing)
- **PC3** → skew (probability mass moving to high-count vs. low-count tail)

*Max useful PCs = min(N_buckets − 1, N_timestamps − 1)*

In [ ]:

from dataclasses import dataclass
from sklearn.decomposition import PCA

# ── Price matrix ────────────────────────────────────────────────────────────
def build_price_matrix(ohlcv_df: pd.DataFrame, market_name: str) -> pd.DataFrame:
    """
    Pivot close prices into (timestamps × bucket_mid) matrix.
    Rows are normalised to sum to 1 so each row is a proper PMF.
    Returns a DataFrame with bucket_mid values as column names.
    """
    sub = ohlcv_df[ohlcv_df['market_name'] == market_name].copy()
    if sub.empty:
        raise ValueError(f"No data for market: {market_name!r}")

    matrix = sub.pivot_table(
        index='timestamp', columns='bucket_mid', values='close', aggfunc='last'
    )
    # Forward-fill small gaps (≤3 periods), then drop rows still too sparse
    matrix = matrix.ffill(limit=3)
    matrix = matrix.dropna(thresh=int(matrix.shape[1] * 0.8))
    matrix = matrix.ffill().bfill()
    # Normalise rows so they sum to 1 (implied PMF)
    matrix = matrix.div(matrix.sum(axis=1), axis=0)
    return matrix


# ── CLR transform ───────────────────────────────────────────────────────────
def clr_transform(prices: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Centered Log-Ratio transform for compositional data.
    Input:  (T, N) — each row is a probability distribution (sums to 1)
    Output: (T, N) — each row sums to 0, rank = N-1
    No information is lost; the simplex constraint is correctly removed.
    """
    p = np.clip(prices, eps, 1.0 - eps)
    log_p = np.log(p)
    return log_p - log_p.mean(axis=1, keepdims=True)


# ── PCA result container ────────────────────────────────────────────────────
@dataclass
class PCAResult:
    market_name:                  str
    n_markets:                    int          # number of buckets
    n_timestamps:                 int
    max_possible_components:      int          # = min(N-1, T-1)
    bucket_midpoints:             np.ndarray
    explained_variance_ratio:     np.ndarray
    cumulative_explained_variance: np.ndarray
    loadings:                     pd.DataFrame  # shape (n_buckets, n_pcs)
    pc_scores:                    pd.DataFrame  # shape (n_timestamps, n_pcs)
    pca_obj:                      object        # fitted sklearn PCA


# ── PCA runner ──────────────────────────────────────────────────────────────
def run_pca(price_matrix: pd.DataFrame, market_name: str,
            n_components: int | None = None) -> PCAResult:
    """
    Apply CLR transform then PCA.
    n_components defaults to max_possible_components = min(N-1, T-1).
    Prints a dimension summary and explained-variance table.
    """
    N = price_matrix.shape[1]   # number of buckets
    T = price_matrix.shape[0]   # number of timestamps
    max_pcs = min(N - 1, T - 1)

    if n_components is None:
        n_components = max_pcs
    n_components = min(n_components, max_pcs)

    prices   = price_matrix.values.astype(float)
    clr_data = clr_transform(prices)

    pca    = PCA(n_components=n_components)
    scores = pca.fit_transform(clr_data)

    pc_cols     = [f"PC{i+1}" for i in range(n_components)]
    bucket_mids = price_matrix.columns.values
    ev_ratio    = pca.explained_variance_ratio_
    cum_ev      = np.cumsum(ev_ratio)

    loadings_df = pd.DataFrame(
        pca.components_.T, index=bucket_mids, columns=pc_cols
    )
    scores_df = pd.DataFrame(
        scores, index=price_matrix.index, columns=pc_cols
    )

    print(f"\n{'='*62}")
    print(f" PCA — {market_name}")
    print(f"{'='*62}")
    print(f"  Timestamps used          : {T:,}")
    print(f"  Buckets/markets used     : {N}")
    print(f"  Bucket column names      : {sorted(bucket_mids)}")
    print(f"  Max possible components  : {max_pcs}  "
          f"[min(N−1={N-1}, T−1={T-1})]")
    print(f"\n  {'PC':<6}  {'Var %':>7}  {'Cumul %':>8}")
    print(f"  {'─'*6}  {'─'*7}  {'─'*8}")
    for i in range(min(15, n_components)):
        marker = " ◄" if cum_ev[i] >= 0.80 and (i == 0 or cum_ev[i-1] < 0.80) else \
                 " ◄" if cum_ev[i] >= 0.95 and (i == 0 or cum_ev[i-1] < 0.95) else ""
        print(f"  PC{i+1:<4}  {ev_ratio[i]*100:>6.2f}%  {cum_ev[i]*100:>7.2f}%{marker}")
    if n_components > 15:
        print(f"  ... ({n_components - 15} more PCs)")

    return PCAResult(
        market_name                  = market_name,
        n_markets                    = N,
        n_timestamps                 = T,
        max_possible_components      = max_pcs,
        bucket_midpoints             = np.array(sorted(bucket_mids)),
        explained_variance_ratio     = ev_ratio,
        cumulative_explained_variance = cum_ev,
        loadings                     = loadings_df,
        pc_scores                    = scores_df,
        pca_obj                      = pca,
    )


print("PCA functions loaded.")
print(f"Available markets in ohlcv_df:")
for m, g in ohlcv_df.groupby('market_name'):
    print(f"  {m:<45}  {g['bucket'].nunique():>3} buckets  {g['timestamp'].nunique()} timestamps")


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 9, 'figure.dpi': 120})


# ── 1. Explained variance ───────────────────────────────────────────────────
def plot_explained_variance(result: PCAResult, max_pcs: int = 20,
                             figsize=(12, 4)) -> plt.Figure:
    """Bar chart of per-PC variance + cumulative line."""
    n   = min(max_pcs, len(result.explained_variance_ratio))
    ev  = result.explained_variance_ratio[:n] * 100
    cev = result.cumulative_explained_variance[:n] * 100
    pcs = [f"PC{i+1}" for i in range(n)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(
        f"PCA Explained Variance — {result.market_name}\n"
        f"({result.n_timestamps} timestamps × {result.n_markets} buckets, "
        f"max {result.max_possible_components} PCs)",
        fontsize=9
    )

    # Per-PC bar
    bars = ax1.bar(pcs, ev, color="steelblue", alpha=0.85, edgecolor="white", linewidth=0.5)
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Explained Variance (%)")
    ax1.set_title("Per-PC Variance")
    ax1.tick_params(axis="x", rotation=60, labelsize=7)
    for bar, v in zip(bars, ev):
        if v > 2:
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                     f"{v:.1f}%", ha="center", va="bottom", fontsize=6)

    # Cumulative line
    ax2.plot(pcs, cev, marker="o", markersize=3, color="steelblue", linewidth=1.5)
    ax2.axhline(80, color="darkorange", linestyle="--", linewidth=0.8, label="80%")
    ax2.axhline(95, color="crimson",    linestyle="--", linewidth=0.8, label="95%")
    ax2.set_xlabel("Principal Component")
    ax2.set_ylabel("Cumulative Explained Variance (%)")
    ax2.set_title("Cumulative Variance")
    ax2.tick_params(axis="x", rotation=60, labelsize=7)
    ax2.set_ylim(0, 105)
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax2.legend(fontsize=8)

    plt.tight_layout()
    return fig


# ── 2. Loadings heatmap ─────────────────────────────────────────────────────
def plot_pca_loadings(result: PCAResult, max_pcs: int = 10,
                      figsize=(12, 5)) -> plt.Figure:
    """Heatmap: rows = PCs, columns = buckets ordered by tweet count."""
    n = min(max_pcs, result.n_markets - 1)
    loadings = result.loadings.sort_index().iloc[:, :n]   # ordered by bucket_mid

    fig, ax = plt.subplots(figsize=figsize)
    vmax = np.abs(loadings.values).max()
    im   = ax.imshow(loadings.values.T, aspect="auto", cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Loading weight", fontsize=8)

    # Y axis: PC labels with variance
    ax.set_yticks(range(n))
    ax.set_yticklabels(
        [f"PC{i+1}  ({result.explained_variance_ratio[i]*100:.1f}%)" for i in range(n)],
        fontsize=7
    )

    # X axis: bucket midpoints (skip every few if crowded)
    bm = loadings.index.values
    step = max(1, len(bm) // 25)
    ax.set_xticks(range(0, len(bm), step))
    ax.set_xticklabels([f"{bm[i]:.0f}" for i in range(0, len(bm), step)],
                       rotation=90, fontsize=7)
    ax.set_xlabel("Bucket midpoint (tweet count)")
    ax.set_ylabel("Principal Component")
    ax.set_title(
        f"PCA Loading Heatmap — {result.market_name}\n"
        f"({result.n_markets} buckets, max {result.max_possible_components} useful PCs)"
    )

    plt.tight_layout()
    return fig


# ── 3. Loading bar charts (first N PCs) ─────────────────────────────────────
def plot_loading_bars(result: PCAResult, n_pcs: int = 3,
                      figsize=(13, 3.5)) -> plt.Figure:
    """One bar chart per PC, ordered by bucket midpoint."""
    n_pcs    = min(n_pcs, result.n_markets - 1)
    loadings = result.loadings.sort_index()
    bm       = loadings.index.values

    fig, axes = plt.subplots(1, n_pcs, figsize=figsize, sharey=False)
    if n_pcs == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        pc   = f"PC{i+1}"
        vals = loadings[pc].values
        cols = ["#d73027" if v > 0 else "#4575b4" for v in vals]
        ax.bar(range(len(vals)), vals, color=cols, alpha=0.85, edgecolor="white", linewidth=0.3)
        ax.axhline(0, color="black", linewidth=0.6)

        step = max(1, len(bm) // 15)
        ax.set_xticks(range(0, len(bm), step))
        ax.set_xticklabels([f"{bm[j]:.0f}" for j in range(0, len(bm), step)],
                           rotation=90, fontsize=7)
        ax.set_xlabel("Bucket midpoint")
        ax.set_ylabel("Loading" if i == 0 else "")
        ax.set_title(f"{pc}  ({result.explained_variance_ratio[i]*100:.1f}%)")

    fig.suptitle(
        f"Loading Bars — {result.market_name}  "
        f"[{result.n_markets} buckets, {result.n_timestamps} timestamps]",
        fontsize=9
    )
    plt.tight_layout()
    return fig


# ── 4. PC scores over time ──────────────────────────────────────────────────
def plot_pc_scores(result: PCAResult, n_pcs: int = 3,
                   rolling_window: int = 12, figsize=(14, 3)) -> plt.Figure:
    """Time series of PC scores + rolling z-score overlay."""
    n_pcs  = min(n_pcs, result.pc_scores.shape[1])
    scores = result.pc_scores.iloc[:, :n_pcs]

    fig, axes = plt.subplots(n_pcs, 1, figsize=(figsize[0], figsize[1] * n_pcs),
                              sharex=True)
    if n_pcs == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        pc = f"PC{i+1}"
        s  = scores[pc]

        ax.plot(s.index, s.values, color="steelblue", linewidth=0.8, alpha=0.9)
        ax.set_ylabel(f"{pc}\nscore", fontsize=8)
        ax.set_title(
            f"{pc}  ({result.explained_variance_ratio[i]*100:.1f}% variance)  —  "
            f"{result.market_name}",
            fontsize=8
        )
        ax.axhline(0, color="gray", linewidth=0.4, linestyle=":")

        # Rolling z-score on twin axis
        roll_mean = s.rolling(rolling_window, min_periods=1).mean()
        roll_std  = s.rolling(rolling_window, min_periods=1).std().fillna(method='bfill').clip(lower=1e-8)
        z = (s - roll_mean) / roll_std

        ax2 = ax.twinx()
        ax2.plot(s.index, z.values, color="darkorange", linewidth=0.6, alpha=0.55,
                 label=f"rolling z (w={rolling_window})")
        ax2.axhline( 2, color="crimson", linestyle="--", linewidth=0.5, alpha=0.5)
        ax2.axhline(-2, color="crimson", linestyle="--", linewidth=0.5, alpha=0.5)
        ax2.set_ylabel("z-score", color="darkorange", fontsize=7)
        ax2.tick_params(axis="y", labelcolor="darkorange", labelsize=7)
        ax2.legend(fontsize=7, loc="upper right")

    axes[-1].set_xlabel("Timestamp (UTC)")
    fig.suptitle(
        f"PC Scores over Time — {result.market_name}  "
        f"[{result.n_timestamps:,} timestamps, "
        f"max {result.max_possible_components} PCs]",
        fontsize=9, y=1.01
    )
    plt.tight_layout()
    return fig


print("Plotting functions loaded: plot_explained_variance, plot_pca_loadings,")
print("                           plot_loading_bars, plot_pc_scores")


In [ ]:

# ── Run PCA for each market in ohlcv_df ────────────────────────────────────
# NOTE: more timestamps = better PCA. Extend DATE_END at the top to get
#       a longer window (e.g. full month gives ~700 hourly timestamps).

pca_results: dict[str, PCAResult] = {}

for market_name in ohlcv_df['market_name'].unique():
    try:
        price_matrix = build_price_matrix(ohlcv_df, market_name)
        if price_matrix.shape[1] < 3:
            print(f"Skipping {market_name!r}: only {price_matrix.shape[1]} buckets (need ≥3)")
            continue
        if price_matrix.shape[0] < 5:
            print(f"Skipping {market_name!r}: only {price_matrix.shape[0]} timestamps (need ≥5)")
            continue
        result = run_pca(price_matrix, market_name)
        pca_results[market_name] = result
    except Exception as e:
        print(f"  ✗ {market_name}: {e}")

print(f"\n\nReady: {len(pca_results)} PCA result(s) in `pca_results` dict")
print("Access a result: pca_results['<market_name>']")


In [ ]:

# ── Visualise: pick the market with the most buckets ───────────────────────
if not pca_results:
    print("No PCA results. Check that ohlcv_df has multiple buckets per market.")
else:
    # Use the market with the most buckets for the richest plot
    target = max(pca_results, key=lambda m: pca_results[m].n_markets)
    r      = pca_results[target]

    print(f"Plotting: {target!r}")
    print(f"  n_markets (buckets):   {r.n_markets}")
    print(f"  n_timestamps:          {r.n_timestamps}")
    print(f"  max_possible_components: {r.max_possible_components}")

    # ── Plot 1: explained variance ─────────────────────────────────────────
    fig1 = plot_explained_variance(r, max_pcs=min(20, r.max_possible_components))
    plt.show()

    # ── Plot 2: loadings heatmap ───────────────────────────────────────────
    fig2 = plot_pca_loadings(r, max_pcs=min(10, r.max_possible_components))
    plt.show()

    # ── Plot 3: loading bars for first 3 PCs ──────────────────────────────
    fig3 = plot_loading_bars(r, n_pcs=min(3, r.max_possible_components))
    plt.show()

    # ── Plot 4: PC score time series ──────────────────────────────────────
    fig4 = plot_pc_scores(r, n_pcs=min(3, r.max_possible_components))
    plt.show()


In [ ]:

# ── Implied moments from distribution ──────────────────────────────────────
# Extract E[tweets] and Var[tweets] from the price matrix as a sanity check.
# These are what the PCs should roughly decompose.

def compute_implied_moments(price_matrix: pd.DataFrame) -> pd.DataFrame:
    """
    Given a (T × N_buckets) price matrix, compute distributional moments.
    bucket_mid column names are the tweet-count midpoints.
    """
    bm = price_matrix.columns.values.astype(float)
    p  = price_matrix.values                         # (T, N)
    ev   = (p * bm).sum(axis=1)                      # E[X]
    ev2  = (p * bm**2).sum(axis=1)                   # E[X²]
    var  = ev2 - ev**2                                # Var[X]
    std  = np.sqrt(np.clip(var, 0, None))

    # Skewness: E[(X-μ)³] / σ³
    e3   = (p * (bm - ev[:, None])**3).sum(axis=1)
    skew = np.where(std > 1e-8, e3 / std**3, 0.0)

    return pd.DataFrame({
        'E_tweets': ev,
        'Std_tweets': std,
        'Skew': skew,
    }, index=price_matrix.index)


if pca_results:
    moments = compute_implied_moments(
        build_price_matrix(ohlcv_df, target)
    )

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(f"Implied Distribution Moments — {target}", fontsize=9)

    axes[0].plot(moments.index, moments['E_tweets'],    color='steelblue',  linewidth=1)
    axes[0].set_ylabel('E[tweets]')
    axes[0].set_title('Expected tweet count (implied by market prices)')

    axes[1].plot(moments.index, moments['Std_tweets'],  color='darkorange', linewidth=1)
    axes[1].set_ylabel('Std[tweets]')
    axes[1].set_title('Uncertainty (std dev) in tweet count')

    axes[2].plot(moments.index, moments['Skew'],        color='purple',     linewidth=1)
    axes[2].axhline(0, color='gray', linewidth=0.5, linestyle=':')
    axes[2].set_ylabel('Skewness')
    axes[2].set_title('Skewness (+ = right tail; – = left tail)')
    axes[2].set_xlabel('Timestamp')

    plt.tight_layout()
    plt.show()
